# A/B Testing

In [33]:
# import modules

import matplotlib.pyplot as plt
import pandas as pd
from statsmodels.stats.proportion import confint_proportions_2indep, proportions_ztest


In [7]:
# read data

df = pd.read_csv('../01_data/02_processed/marketing_AB_cleaned.csv')

In [ ]:
df.head(5)

,user_id,test_group,converted,total_ads,most_ads_day,most_ads_hour,test_group_num,weekend,time_of_day
0,1069124,ad,0,130,Monday,20,1,0,evening
1,1119715,ad,0,93,Tuesday,22,1,0,evening
2,1144181,ad,0,21,Tuesday,18,1,0,evening
3,1435133,ad,0,355,Tuesday,10,1,0,midday
4,1015700,ad,0,276,Friday,14,1,0,afternoon


### Primary question

Does the recorded conversion rate differ between users assigned to the
advertising (`ad`) group and users assigned to the PSA (`psa`) group?

### Primary metric

Recorded conversion rate:

Conversion rate = number of converted users / number of users in the group

### Groups

- **Control:** `psa`
- **Treatment:** `ad`

### Hypotheses

- **H₀:** The recorded conversion rate is equal in the ad and PSA groups.
- **H₁:** The recorded conversion rate differs between the ad and PSA groups.

### Statistical approach

The primary analysis uses a two-sided two-proportion z-test at a 5% significance level.
A 95% confidence interval is reported for the absolute difference in
conversion rates.

### Decision rule

The result is considered statistically significant when:

- the two-sided p-value is below 0.05, and
- the 95% confidence interval for the absolute lift does not include zero.

Statistical significance alone does not establish commercial value.
Business relevance will be assessed separately through scenario analysis.


## Test groups and conversion

In [ ]:
df.groupby('test_group').agg({'converted': 'sum', 'user_id': 'count'})

,converted,user_id
test_group,,
ad,14423,564577
psa,420,23524


In [22]:
# conversion rate of the two groups

conversion_rate_psa = df[df['test_group'] == 'psa']['converted'].sum() / df[df['test_group'] == 'psa'].shape[0]
conversion_rate_ad = df[df['test_group'] == 'ad']['converted'].sum() / df[df['test_group'] == 'ad'].shape[0]
print("Conversion rate (psa):", conversion_rate_psa)
print("Conversion rate (ad):", conversion_rate_ad)

Conversion rate (psa): 0.01785410644448223
Conversion rate (ad): 0.025546559636683747


## Effect size: absolute and relative lift

### Effect-size definitions

- The absolute lift is the difference in conversion rates: Absolute lift = Conversion rate (ad) − Conversion rate (PSA)
- The relative lift expresses the difference relative to the PSA control rate:
- Relative lift = (Conversion rate (ad) − Conversion rate (PSA))
                ÷ Conversion rate (PSA)

In [23]:
# Effect sizes

lift_absolute = conversion_rate_ad - conversion_rate_psa
lift_relative = (conversion_rate_ad - conversion_rate_psa) / conversion_rate_psa

print("Absolute lift:", lift_absolute)
print("Relative lift:", lift_relative)

Absolute lift: 0.007692453192201517
Relative lift: 0.43085064022225833


### Observed effect

- The ad group recorded a conversion rate of 2.555%, compared with 1.785% in the PSA control group.
- This corresponds to an observed absolute lift of **0.769 percentage points** and a relative lift of **43.1%**.
- The absolute lift is the primary quantity for the later scenario analysis: for every 100,000 users reached, a lift of 0.769 percentage points would correspond to approximately 769 additional recorded conversions, assuming the observed difference is causal and generalises to the target population.

## Uncertainty: 95% confidence interval

### Confidence interval for the absolute lift

A positive value indicates a higher recorded conversion rate in the ad group. A 95% score confidence interval is calculated for the difference between two independent proportions.

The interval quantifies uncertainty around the estimated absolute lift. If the complete interval lies above zero, the data are inconsistent with no difference at the 5% significance level.

In [30]:
ad_users =  df[df['test_group'] == 'ad'].shape[0]
ad_conversions = df[df['test_group'] == 'ad']['converted'].sum()
psa_users = df[df['test_group'] == 'psa'].shape[0]
psa_conversions = df[df['test_group'] == 'psa']['converted'].sum()


ci_low, ci_high = confint_proportions_2indep(
    count1=ad_conversions,
    nobs1=ad_users,
    count2=psa_conversions,
    nobs2=psa_users,
    compare="diff",
    method="score",
    alpha=0.05
)

print(
    f"95% CI for absolute lift: "
    f"{ci_low * 100:.3f} to {ci_high * 100:.3f} percentage points"
)

95% CI for absolute lift: 0.587 to 0.936 percentage points


### Confidence-interval interpretation

The estimated absolute conversion lift is **0.769 percentage points**.

The 95% confidence interval ranges from **0.587 to 0.936 percentage
points**. Because the full interval lies above zero, the observed
difference is statistically significant at the 5% level.

The interval also provides a range for scenario planning:

- **Conservative case:** 0.587 percentage points
- **Base case:** 0.769 percentage points
- **Upside case:** 0.936 percentage points

These values are not guaranteed future outcomes. They represent
data-informed assumptions conditional on the original assignment being
valid and on the effect generalising to a future campaign population.

## Hypothesis Testing

#### Two-proportion z-test

The primary hypothesis test compares the recorded conversion rates in the
ad and PSA groups.

- **H₀:** Conversion rate (ad) = Conversion rate (PSA)
- **H₁:** Conversion rate (ad) ≠ Conversion rate (PSA)

A two-sided test is used with a significance level of 5%.

In [35]:
z_stat, p_value = proportions_ztest(
    count=[ad_conversions, psa_conversions],
    nobs=[ad_users, psa_users],
    alternative="two-sided"
)

print(f"Z-statistic: {z_stat:.3f}")
print(f"P-value: {p_value:.3e}")

Z-statistic: 7.370
P-value: 1.705e-13


### Hypothesis-test result

The two-sided two-proportion z-test returned a test statistic of **z = 7.37** and a p-value below **0.001**.

The positive z-statistic indicates that the recorded conversion rate was higher in the ad group than in the PSA group. The observed difference is 7.37 standard errors above the no-difference value of zero.

At the pre-specified 5% significance level, the null hypothesis of equal conversion rates is rejected. The result is consistent with a positive difference in recorded conversion between the ad and PSA groups.

This statistical result does not, on its own, establish that a future campaign will generate the same effect or that the effect is commercially worthwhile.

### Effect size (absolute and relative)

The observed absolute difference in recorded conversion rates is +0.769 percentage points (from 1.785% to 2.555%). The relative lift
compared to the PSA group is approximately 43%.

Expressed on a scale of 100,000 reached users, this corresponds to:

- PSA: ~1,785 conversions
- Ad:  ~2,555 conversions
- Additional conversions: ~770 per 100,000 reached users

This absolute effect size is the relevant quantity for estimating business impact (e.g. additional leads, sign-ups, or purchases), while
the relative lift is useful for comparing effects across different baselines or experiments.

# Business impact and practical significance

This section translates the observed effect size into an estimated business impact. It uses the recorded conversion rates and the absolute
lift as inputs, and explores how different assumptions about the value per conversion and the cost of reaching users affect the decision to run the campaign.